# Lab 3 — one controlled change and a defensible recommendation

100 points · 6% NP · same pairs as Lab 2 · due Thu 29 October 2026, 12:30 Santiago.

Lab 1 checked data and simple rules. Lab 2 changed one input under fixed comparison conditions.
Here you keep the inputs fixed and change one model setting, then explain errors and write for a
Head of Strategy. The supplied code runs the experiment; you must supply the checked explanation,
one-change specification, three supported failure patterns, memo and truthful AI record.

Follow the numbered headings. See README for setup and GLOSSARY for terms.
A losing additional candidate can earn full credit. The unchanged starter is incomplete.


## Environment and data check

Extract the whole ZIP and choose your kernel before running this block. Follow README.md if a check fails.

In [1]:
from pathlib import Path
import sys
import importlib
print('Selected Python:', sys.executable)
print('Version:', sys.version.split()[0], '| working directory:', Path.cwd())
required_files = ['lab03_support_v3.py', 'pitwindow_student_v2.py', 'data/learning_v2.csv']
missing = [name for name in required_files if not Path(name).is_file()]
if missing:
    raise FileNotFoundError('Extract the complete ZIP and open its folder as cwd. Missing: ' + ', '.join(missing))
for module in ['numpy', 'pandas', 'sklearn', 'scipy', 'joblib']:
    loaded = importlib.import_module(module)
    print(module, loaded.__version__)
print('Ready: local files/imports found. Next cell checks data integrity and split.')


Selected Python: /home/remplenque/anaconda3/bin/python
Version: 3.13.5 | working directory: /home/remplenque/Documents/UDD/LAB 3_AI_WORKSHOP/IIT414W_Lab03_StudentPack_v3
numpy 2.1.3


pandas 2.2.3


sklearn 1.6.1
scipy 1.15.3
joblib 1.4.2
Ready: local files/imports found. Next cell checks data integrity and split.


In [2]:
from pathlib import Path
import pandas as pd
from lab03_support_v3 import (
    FEATURES, SEED, BUDGET, load_learning, run_base_comparison,
    run_one_change, export_run, markdown_table,
)
ROOT = Path('.')
train, dev = load_learning(ROOT)
print('Seed:', SEED, '| development alert budget:', BUDGET)
print('Permitted inputs:', FEATURES)
audit = pd.DataFrame([
    {'split': name, 'rows': len(df), 'positive_windows': int(df.y.sum()),
     'eligible_events': df.loc[df.y.eq(1), 'event_id'].nunique(),
     'first_year': int(df.year.min()), 'last_year': int(df.year.max())}
    for name, df in [('train', train), ('dev', dev)]
])
display(audit)
assert set(train.opportunity_id).isdisjoint(dev.opportunity_id)


Seed: 414 | development alert budget: 0.1
Permitted inputs: ['circuit_id', 'compound', 'tyre_life', 'lap_number', 'progress', 'prior_pit_entries', 'pace_median_s', 'pace_slope_s_per_lap', 'pace_n']


,split,rows,positive_windows,eligible_events,first_year,last_year
0,train,63165,5523,1934,2018,2021
1,dev,9451,1073,370,2022,2022


## 1. Define the prediction

**When we predict.** At the end of a rival driver's lap L (the `cut_s` time of that lap). Each row is one opportunity (window) for one driver in one race, and the question is always about that same driver, not about our own car.

**What y = 1 means.** That driver enters the pit lane during their own next three laps (L+1 to L+3). It is an observed entry, so it says nothing about whether stopping was the right call. Because the horizon is three laps, one pit entry normally makes up to three consecutive windows positive: in train, 1741 of the 1934 events have three positive windows (check cell below). That is why we keep windows and events (distinct pit entries, `event_id`) apart in everything that follows.

**One allowed input: `tyre_life`.** Laps done on the current set of tyres at the end of lap L. It only looks backwards, so it exists at prediction time. One caveat from the glossary: it is the last *reported* value and can lag, so live it could be one lap old.

**One future-known column kept out of X: `future_neutralisation`.** It flags whether a neutralisation (safety car / VSC type period) happens inside the three-lap horizon. At the end of lap L nobody knows that yet. It would also be a strong but fake signal: in train, y = 1 in 34.7% of windows where it is True and in 7.5% where it is False (check cell). If it went into X the dev results would look much better than anything we could get live. The same applies to `event_id` and `event_s` (they contain the actual pit entry time) and `horizon_end_s` (when lap L+3 ends). These columns are only used to build the label and to count events; the models only receive the nine `FEATURES` printed above.

**What this retrospective snapshot can and cannot establish.** The CSV was built after the races from public FastF1 timing and was corrected afterwards. It can show that, inside the historical record, each feature is computed from laps up to L, so the ordering of information is right. It cannot show that those values would really have arrived by the end of lap L in a live feed (delays, missing laps, later revisions of tyre data), and it is not a prospective test. There is also a time gap built into the split: models learn on 2018–2021 and are compared on the first 11 rounds of 2022, the first season with the new cars and 18-inch tyres. Compound names are not the same either (train still has 2018 ULTRASOFT/SUPERSOFT/HYPERSOFT, dev only has SOFT/MEDIUM/HARD/INTERMEDIATE/WET), and circuit `151` (Miami, 8 May 2022) appears in dev with zero training rows.


In [3]:
# Quick check behind the numbers in section 1 (train rows only, descriptive, nothing is fitted here).
# future_neutralisation is looked at ONLY to show why it must stay out of X.
print('Positive rate by future_neutralisation (train):')
print(train.groupby('future_neutralisation').y.agg(['mean', 'size']).round(3))
print('\nPositive windows per event (train): windows -> number of events')
print(train.loc[train.y.eq(1)].groupby('event_id').size().value_counts().sort_index())
print('\nDev circuits with no training rows:', sorted(set(dev.circuit_id) - set(train.circuit_id)))


Positive rate by future_neutralisation (train):
                        mean   size
future_neutralisation              
False                  0.075  60329
True                   0.347   2836

Positive windows per event (train): windows -> number of events
1      86
2     107
3    1741
Name: count, dtype: int64

Dev circuits with no training rows: ['151']


## Practice reading the counts

Illustrative manual example only, NOT F1 data. Two positive windows refer to event A; a third refers
to event B. The final window is negative and triggers a false alert.

| Window | y | Event | Alert |
|---|---:|---|---|
| w1 | 1 | A | yes |
| w2 | 1 | A | yes |
| w3 | 1 | B | no |
| w4 | 0 | none | yes |

Ungraded check: count TP, FP, FN, alerts, eligible/caught events, precision and event recall.
What changes if w2 stops alerting? Think first, then open the self-check below.


### Optional practice response

TP = 2 (w1, w2); FP = 1 (w4); FN = 1 (w3); alerts = 3; eligible/caught events = 2 / 1 (A caught, B not); precision = 2/3; event recall = 1/2. If w2 becomes quiet: TP = 1, FN = 2, alerts = 2, precision = 1/2, but event recall stays 1/2 because w1 still catches A.

<details>
<summary>Open after trying: self-check for the four fictional windows</summary>

TP=2, FP=1, FN=1, TN=0; alerts=3; precision=2/3. There are two eligible events A/B, one caught, so event recall=1/2. If w2 becomes quiet: TP=1, FP=1, FN=2, alerts=2, precision=1/2; A is still caught by w1 and event recall remains 1/2.

</details>

In [4]:
base_comparison, fitted, base_policies = run_base_comparison(train, dev)
compact_columns = ['model', 'alerts', 'precision', 'events_caught', 'eligible_events', 'event_recall']
print('Start with DEVELOPMENT. Undefined precision means there were no alerts.')
display(base_comparison.loc[base_comparison.split.eq('dev'), compact_columns])
columns = ['model', 'split', 'n', 'TP', 'FP', 'FN', 'TN', 'alerts', 'alert_rate',
           'precision', 'events_caught', 'eligible_events', 'event_recall', 'threshold']
print("Audit view: train and development reuse the threshold selected for that model on dev.")
display(base_comparison[columns])


Start with DEVELOPMENT. Undefined precision means there were no alerts.


,model,alerts,precision,events_caught,eligible_events,event_recall
1,never_alert,0,NaN,0,370,0.000000
3,historical_baseline,931,0.268528,94,370,0.254054
5,tree,571,0.269702,63,370,0.170270
7,forest_all,938,0.276119,107,370,0.289189
9,forest_sqrt,945,0.359788,140,370,0.378378


Audit view: train and development reuse the threshold selected for that model on dev.


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
0,never_alert,train,63165,0,0,5523,57642,0,0.000000,NaN,0,1934,0.000000,NaN
1,never_alert,dev,9451,0,0,1073,8378,0,0.000000,NaN,0,370,0.000000,NaN
2,historical_baseline,train,63165,3042,6645,2481,50997,9687,0.153360,0.314029,1137,1934,0.587901,0.161926
3,historical_baseline,dev,9451,250,681,823,7697,931,0.098508,0.268528,94,370,0.254054,0.161926
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
6,forest_all,train,63165,2503,4503,3020,53139,7006,0.110916,0.357265,969,1934,0.501034,0.718176
7,forest_all,dev,9451,259,679,814,7699,938,0.099249,0.276119,107,370,0.289189,0.718176
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283


## 2. Explain the common comparison

Dev: 9451 windows, 1073 positive windows, 370 eligible entries, at most 945 alerts (10%) for every system.

**forest_all vs forest_sqrt.** Both use almost the whole budget (938 and 945 alerts), so the difference is in ranking, not in alerting more. forest_sqrt catches 140/370 entries (event recall 0.378) vs 107/370 (0.289), with 340 TP / 605 FP (precision 0.360) vs 259 TP / 679 FP (0.276). AP points the same way (0.331 vs 0.236). For reference, the baseline catches 94/370 with 931 alerts and the tree 63/370 with 571.

**What max_features changes.** The two forests are identical except `max_features`. After preprocessing there are 48 columns: 7 numeric, 3 missing-value indicators, 29 circuit and 9 compound dummies (check cell below). forest_all evaluates all 48 at every split; forest_sqrt draws a fresh random set of int(√48) = 6 at every split. Nothing is removed for good: a column skipped at one split can be used at the next one. With all 48 available, every tree tends to take the same strong splits (our depth-3 tree starts with `prior_pit_entries`, `progress`, `tyre_life`), so trees only differ by their bootstrap sample. With 6, about 37% of draws contain no raw numeric column at all (C(41,6)/C(48,6)), which forces other splits. We think this makes the trees more varied and less tied to the exact 2018–2021 cut points, but we did not measure correlation between trees, so that part is a hypothesis. What we can see is consistent with it: forest_sqrt has a lower train AP (0.310 vs 0.349) but a higher dev AP (0.331 vs 0.236).

**Undefined precision.** never_alert has 0 alerts, so TP/alerts = 0/0, which is undefined, not 0%. Its event recall is still defined: 0/370.

**TP windows vs caught events.** forest_sqrt has 340 TP windows but 140 caught entries, because an entry counts once if any of its (usually 3) positive windows alerts. In the same way its 733 FN windows are not 733 misses: 230 entries got no alert at all and the other FN windows belong to entries caught on another lap. Precision is per alert (945), event recall per entry (370).

**Same policy, different thresholds.** The rule is common (≤ 945 alerts, most entries, then precision, then threshold) but each model has its own score scale: 0.608 (forest_sqrt), 0.718 (forest_all), 0.714 (tree), 0.162 (baseline, a smoothed historical rate; balanced class weights push the learned models' scores up). None of them is a calibrated probability. Alert counts differ for mechanical reasons too. The tree only has 8 possible scores and its next tied block would jump from 571 to 1898 alerts, so 374 alerts go unused. forest_all stops at 938 because windows 939–945 are all false and add no entry (check cell), so the precision tie-break keeps the smaller set.

**One train/dev gap (forest_sqrt).** Event recall 1005/1934 = 0.520 on train vs 140/370 = 0.378 on dev. Train is in-sample and dev chose the threshold, so neither is a clean test. The reused threshold also fires on 14.4% of train windows vs 10% on dev, and prevalence differs (8.7% vs 11.4% positive windows, which is why dev precision is even higher, 0.360 vs 0.294). So the gap fits overfitting and/or the 2022 change, but proves neither. forest_all's gap is larger (0.501 → 0.289) and the baseline's the largest (0.588 → 0.254).


In [5]:
# Checks behind section 2 (base models only, nothing refitted or retuned)
import numpy as np
forest_sqrt_model = fitted['forest_sqrt'].named_steps['model']
n_cols = forest_sqrt_model.n_features_in_
print('Columns after preprocessing:', n_cols, '| forest_sqrt candidates per split:', int(np.sqrt(n_cols)))
print(list(fitted['forest_sqrt'].named_steps['prep'].get_feature_names_out())[:10], '... + circuit/compound dummies')

# Why forest_all used 938 alerts and not 945
scores_all = fitted['forest_all'].predict_proba(dev[FEATURES])[:, 1]
order = np.argsort(-scores_all, kind='stable')
for k in [938, 945]:
    top = dev.iloc[order[:k]]
    print(f'forest_all top {k} windows: TP={int(top.y.sum())}, entries caught={top.loc[top.y.eq(1), "event_id"].nunique()}')


Columns after preprocessing: 48 | forest_sqrt candidates per split: 6
['numeric__tyre_life', 'numeric__lap_number', 'numeric__progress', 'numeric__prior_pit_entries', 'numeric__pace_median_s', 'numeric__pace_slope_s_per_lap', 'numeric__pace_n', 'numeric__missingindicator_tyre_life', 'numeric__missingindicator_pace_median_s', 'numeric__missingindicator_pace_slope_s_per_lap'] ... + circuit/compound dummies
forest_all top 938 windows: TP=259, entries caught=107
forest_all top 945 windows: TP=259, entries caught=107


## 3. Record your expectation before your additional run

The base outputs are already supplied. Complete `experiment_spec.md` before running YOUR candidate
for the first time. If already run, say so and keep a retrospective account. Do not invent history.

Choose ONE tuple: tree / max_depth / 6 (old 3); forest_sqrt / max_depth / 3 (old 6);
or forest_sqrt / min_samples_leaf / 100 (old 30). No search for the best of all three.
The helper clones the base pipeline and changes one setting. Its threshold is reselected on dev
under the same policy, so the numeric threshold may differ. This compares the resulting alert systems.


Choose one additional configuration. Repeating that same configuration for debugging or Restart + Run All is allowed. Disclose any different configurations you already tried; do not hide a search.

In [6]:
# Parent tree only (already fitted base model, no new fit): why we picked tree / max_depth / 6.
# Depth 3 -> at most 8 leaves -> only 8 different scores. Each block is all-or-nothing for the threshold.
import numpy as np
parent_scores = fitted['tree'].predict_proba(dev[FEATURES])[:, 1]
blocks = (pd.DataFrame({'score': parent_scores, 'y': dev.y})
          .groupby('score').agg(windows=('y', 'size'), positive_windows=('y', 'sum'))
          .sort_index(ascending=False))
blocks['cumulative_alerts'] = blocks.windows.cumsum()
print('Dev alert cap:', int(np.floor(0.10 * len(dev))))
display(blocks)


Dev alert cap: 945


,windows,positive_windows,cumulative_alerts
score,,,
0.836733,361,83.0,361
0.713735,210,71.0,571
0.651749,1327,242.0,1898
0.573988,1960,288.0,3858
0.551606,134,3.0,3992
0.408171,973,142.0,4965
0.283905,999,35.0,5964
0.179757,3487,209.0,9451


In [7]:
EXPERIMENT_MODEL_ID = 'tree'         # 'tree' or 'forest_sqrt', matching the allowed table
EXPERIMENT_PARAMETER = 'max_depth'   # 'max_depth' or 'min_samples_leaf'
EXPERIMENT_VALUE = 6                 # the one allowed new value (parent tree uses 3)


In [8]:
change = None
policies = dict(base_policies)
comparison = base_comparison.copy()
if any(value is None for value in [EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE]):
    print('INCOMPLETE STARTER: configure one additional experiment in extra-config.')
else:
    extra_rows, extra_model, extra_policy, change = run_one_change(
        train, dev, fitted, EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE)
    comparison = pd.concat([base_comparison, extra_rows], ignore_index=True)
    policies[change['model_id']] = extra_policy
    display(comparison.loc[comparison.model.isin([change['parent'], change['model_id']]), columns])


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
10,tree__max_depth_6,train,63165,2068,3741,3455,53901,5809,0.091965,0.355999,775,1934,0.400724,0.790333
11,tree__max_depth_6,dev,9451,145,435,928,7943,580,0.061369,0.250000,57,370,0.154054,0.790333


## 4. Explain your additional result

**What happened (dev, 370 entries, 945-alert cap).** `tree__max_depth_6` caught 57/370 entries (0.154) vs 63/370 (0.170) for the parent tree, with 580 vs 571 alerts and 145 TP / 435 FP, so precision fell from 0.270 to 0.250 (threshold 0.790 vs 0.714). On train it is the opposite: event recall 0.336 → 0.401 (650 → 775 of 1934) and precision 0.273 → 0.356, with fewer train alerts (5809 vs 6647), so that is not just more alerting. The train/dev gap grows from 0.166 to 0.247. Dev AP goes up a little (0.180 → 0.200), so ranking over all dev windows is slightly better, but the policy only uses the top ≤ 10% and that is where it got worse.

**Did our expectation hold?** Mostly no. We expected close to 945 alerts, 85–115 entries and similar or better precision, and all three were wrong. The risk we wrote down, a bigger train/dev gap, did happen. The original expectation stays in `experiment_spec.md`.

**Why (check cell below, frozen candidate, nothing refitted):**
- *The budget is still mostly unused.* 58 leaves instead of 8, but dev windows only reach 36 of them and the top 8 score blocks hold just 580 windows. The next block (score 0.736) has 794 windows by itself, so one step lower means 1374 > 945 alerts and 365 alerts stay unused. We assumed "more leaves = finer steps" and ignored how uneven the leaves could be on 2022 data.
- *The top scores now come from small leaves built on one or two training races.* With min_samples_leaf 50, depth 6 can carve out leaves of ~50 train windows. Three alerted leaves are like that and give 145 of the 580 alerts with only 3 TP: leaf 113 (56 train windows, 53 from 2021-06-06) fires on 51 Baku 2022 windows, 0 positive; leaf 83 (73 windows, 55 from 2020-09-13) on 54 windows at circuit 149, 3 positive; leaf 18 (50 windows, 48% on wet tyres) on 40 lap-2 windows in Bahrain and Miami, 0 positive. Leaving those three out, the other 435 alerts have 142 TP (precision 0.33), so the loss is concentrated in those pockets. That fits a better fit on train and a worse dev result.
- *It reshuffles more than it adds.* 35 entries are caught by both trees, 28 only by the parent and 22 only by the candidate.

**Observed vs hypothesis.** Leaf sizes, source races and dev counts are measured. Why those races were special is a hypothesis: 62% of leaf 113's train windows had a neutralisation in the horizon, which suggests stops caused by a late safety car or red flag, but we did not check the race events (`future_neutralisation` is only used to describe the leaf, never as an input). Each tree also has its own dev threshold, so we are comparing two alert systems, not the effect of `max_depth` alone.

**Recommendation and limits.** We would not use the depth-6 tree: fewer entries and more false alerts than its parent, and far behind forest_sqrt (140/370 with 945 alerts), which is still the system worth checking further. All of these are dev numbers and dev also chose the thresholds, so they are optimistic and not a final test; the data is a corrected historical snapshot, so they also assume inputs would arrive on time live. Finally, 57 vs 63 is only 6 entries over 11 races, so the opposite direction on train and dev is the clearer signal, not the size of the drop.


In [9]:
# Looking inside the frozen candidate to explain section 4 (no refit, no new threshold).
assert change is not None, "Run the additional candidate (extra-config / extra-run) first."
# (a) why it still only uses 580 alerts: dev score blocks of the deeper tree
cand_tree = extra_model.named_steps['model']
cand_dev_scores = extra_model.predict_proba(dev[FEATURES])[:, 1]
dev_leaf = cand_tree.apply(extra_model.named_steps['prep'].transform(dev[FEATURES]))
train_leaf = cand_tree.apply(extra_model.named_steps['prep'].transform(train[FEATURES]))
print('Leaves: parent', fitted['tree'].named_steps['model'].get_n_leaves(),
      '| candidate', cand_tree.get_n_leaves(), '| candidate leaves reached by dev windows', len(np.unique(dev_leaf)))
cand_blocks = (pd.DataFrame({'score': cand_dev_scores, 'y': dev.y})
               .groupby('score').agg(windows=('y', 'size'), positive_windows=('y', 'sum'))
               .sort_index(ascending=False))
cand_blocks['cumulative_alerts'] = cand_blocks.windows.cumsum()
print('Selected threshold:', round(extra_policy['threshold'], 6), '| dev alert cap: 945')
display(cand_blocks.head(10))

# (b) the alerted leaves: how many training windows they were learned from, and which races
rows = []
for leaf in np.unique(dev_leaf[cand_dev_scores >= extra_policy['threshold']]):
    tr_rows, dv_rows = train[train_leaf == leaf], dev[dev_leaf == leaf]
    top_race = tr_rows.race_id.value_counts()
    rows.append(dict(leaf=leaf, score=round(cand_dev_scores[dev_leaf == leaf][0], 4),
                     train_windows=len(tr_rows), train_positive=int(tr_rows.y.sum()),
                     train_races=tr_rows.race_id.nunique(),
                     top_train_race=top_race.index[0], windows_from_top_race=int(top_race.iloc[0]),
                     # future_neutralisation is NEVER an input; used here only to describe the leaf
                     train_share_future_neutralisation=round(tr_rows.future_neutralisation.mean(), 2),
                     train_share_wet_tyres=round(tr_rows.compound.isin(['INTERMEDIATE', 'WET']).mean(), 2),
                     dev_windows=len(dv_rows), dev_positive=int(dv_rows.y.sum()),
                     dev_races=', '.join(sorted(dv_rows.race_id.unique()))))
leaf_table = pd.DataFrame(rows).sort_values('score', ascending=False)
display(leaf_table)

# decision rule of the small alerted leaves (< 100 training windows)
names = extra_model.named_steps['prep'].get_feature_names_out()
t = cand_tree.tree_
parent_of = {child: (node, side) for node in range(t.node_count)
             for child, side in [(t.children_left[node], '<='), (t.children_right[node], '>')] if child != -1}
for leaf in leaf_table.loc[leaf_table.train_windows < 100, 'leaf']:
    rule, node = [], leaf
    while node in parent_of:
        node, side = parent_of[node]
        rule.append(f'{names[t.feature[node]]} {side} {t.threshold[node]:.3f}')
    print(f'leaf {leaf}:', ' AND '.join(reversed(rule)))
print('Share of train windows on INTERMEDIATE/WET:', round(train.compound.isin(['INTERMEDIATE', 'WET']).mean(), 3))

# (c) entries caught by parent vs candidate (same dev rows, each with its own threshold)
parent_alert = fitted['tree'].predict_proba(dev[FEATURES])[:, 1] >= policies['tree']['threshold']
cand_alert = cand_dev_scores >= extra_policy['threshold']
caught_parent = set(dev.loc[parent_alert & dev.y.eq(1), 'event_id'])
caught_cand = set(dev.loc[cand_alert & dev.y.eq(1), 'event_id'])
print('Entries caught: parent', len(caught_parent), '| candidate', len(caught_cand),
      '| both', len(caught_parent & caught_cand), '| only parent', len(caught_parent - caught_cand),
      '| only candidate', len(caught_cand - caught_parent))


Leaves: parent 8 | candidate 58 | candidate leaves reached by dev windows 36
Selected threshold: 0.790333 | dev alert cap: 945


,windows,positive_windows,cumulative_alerts
score,,,
0.941621,51,0.0,51
0.932298,115,43.0,166
0.890659,54,3.0,220
0.890501,3,3.0,223
0.852841,112,56.0,335
0.843514,201,36.0,536
0.843174,40,0.0,576
0.790333,4,4.0,580
0.736467,794,180.0,1374


,leaf,score,train_windows,train_positive,train_races,top_train_race,windows_from_top_race,train_share_future_neutralisation,train_share_wet_tyres,dev_windows,dev_positive,dev_races
7,113,0.9416,56,34,2,2021-06-06,53,0.62,0.00,51,0,2022-06-12
4,56,0.9323,334,190,24,2018-09-02,44,0.03,0.00,115,43,"2022-03-27, 2022-04-10, 2022-05-08, 2022-06-12..."
5,83,0.8907,73,32,2,2020-09-13,55,0.23,0.00,54,3,2022-03-27
2,41,0.8905,274,120,34,2021-05-09,27,0.01,0.00,3,3,"2022-03-20, 2022-05-22"
6,100,0.8528,647,231,4,2020-11-29,235,0.00,0.00,112,56,2022-03-20
3,48,0.8435,2807,956,51,2020-07-12,135,0.05,0.02,201,36,"2022-03-20, 2022-03-27, 2022-04-10, 2022-05-08..."
0,18,0.8432,50,17,6,2020-07-19,16,0.06,0.48,40,0,"2022-03-20, 2022-05-08"
1,40,0.7903,603,160,37,2020-07-12,43,0.06,0.00,4,4,"2022-03-20, 2022-05-22"


leaf 113: numeric__prior_pit_entries > 0.500 AND numeric__tyre_life > 11.500 AND numeric__progress > 0.746 AND categorical__circuit_id_144 > 0.500 AND categorical__compound_MEDIUM <= 0.500 AND numeric__pace_slope_s_per_lap > -0.132
leaf 83: numeric__prior_pit_entries > 0.500 AND numeric__tyre_life <= 11.500 AND categorical__circuit_id_149 > 0.500 AND numeric__lap_number > 23.500 AND numeric__progress <= 0.746 AND numeric__prior_pit_entries <= 2.500
leaf 18: numeric__prior_pit_entries <= 0.500 AND numeric__progress <= 0.287 AND numeric__pace_median_s > 91.501 AND numeric__pace_n <= 2.500 AND numeric__progress <= 0.035
Share of train windows on INTERMEDIATE/WET: 0.044
Entries caught: parent 63 | candidate 57 | both 35 | only parent 28 | only candidate 22


## 5. Explain three distinct failure patterns

Keep your additional candidate and its selected threshold frozen. Inspect the supplied tables, choose the denominator for your claim, and use the queries to find supporting cases. This step does not train or retune.

### Choose your denominator

| Question within your context | Calculation | Denominator counts |
|---|---|---|
| Of the alerts issued, how many were false? | FP / (TP + FP) | Alerted windows; false-alert fraction, equal to 1 − precision when defined. |
| Of the negative windows, how many alerted? | FP / (FP + TN) | Negative windows; false positive rate (FPR). |
| Of the positive windows, how many were quiet? | FN / (TP + FN) | Positive windows; `miss_rate` in the context export. |
| Of the eligible events, how many were completely missed? | Completely missed events / eligible events | Distinct pit entries; use all positive windows for each event. |

A zero denominator means undefined, shown as NaN; it does not mean zero errors. Name your population and unit with every fraction. Window context groups do not define an event population automatically: if describing missed events in a context, state which distinct events you include and check all their positive windows, including those outside that context. The event query below checks a whole event.


### A complete entry from the four fictional windows

This uses only the four-window practice example, not the Rival dataset or an additional candidate's results.

“For the context of all four toy windows, w4 is a false alert (y=0, alert=yes). One of the three alerts is false: FP/alerts=1/3. The traceable case is `opportunity_id=w4`. This could prompt an unnecessary review of a rival that does not enter within the horizon; next check the time and outcome around w4 before attributing a cause. With only four invented windows, this illustrates the reasoning format and cannot establish a general effect.”

The same counts give FPR=1/(1+0)=1/1 over negative windows, and miss_rate=1/(2+1)=1/3 over positive windows. Those fractions answer different questions. Event B has one positive window (w3), none alerted: one of two eligible events is completely missed. A quiet w2 alone would not make event A missed because w1 still alerts.


In [10]:
from lab03_support_v3 import development_predictions, context_error_summary, event_error_summary
predictions = None
if change is None:
    print('INCOMPLETE: run your additional candidate before examining its errors.')
else:
    predictions = development_predictions(dev, extra_model, extra_policy, change['model_id'])
    context_errors = context_error_summary(predictions)
    event_errors = event_error_summary(predictions)
    print('Descriptive context tables. No refit or threshold change occurs here.')
    display(context_errors)
    print('FP window examples (highest scores first):')
    display(predictions.loc[predictions.error_type.eq('FP')].sort_values('score', ascending=False).head(8))
    print('FN window examples (lowest scores first):')
    display(predictions.loc[predictions.error_type.eq('FN')].sort_values('score').head(8))
    print('Missed events:', int(event_errors.missed_event.sum()), '/', len(event_errors))
    display(event_errors.loc[event_errors.missed_event].head(8))


Descriptive context tables. No refit or threshold change occurs here.


,dimension,context,n,positive_windows,alerts,TP,FP,FN,TN,precision,miss_rate
0,compound,HARD,4037,266,394,60,334,206,3437,0.152284,0.774436
1,compound,INTERMEDIATE,377,86,0,0,0,86,291,NaN,1.000000
2,compound,MEDIUM,3593,473,116,54,62,419,3058,0.465517,0.885835
3,compound,SOFT,1211,201,70,31,39,170,971,0.442857,0.845771
4,compound,UNKNOWN,4,0,0,0,0,0,4,NaN,NaN
5,compound,WET,229,47,0,0,0,47,182,NaN,1.000000
6,circuit_id,10,797,61,60,10,50,51,686,0.166667,0.836066
7,circuit_id,144,720,88,60,9,51,79,581,0.150000,0.897727
8,circuit_id,149,617,55,106,10,96,45,466,0.094340,0.818182
9,circuit_id,15,1032,159,4,4,0,155,873,1.000000,0.974843


FP window examples (highest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
6051,2022-06-12:10:40:8299.781000,2022-06-12,10,40,144,HARD,30.0,0.784314,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6522,2022-06-12:47:41:8485.687000,2022-06-12,47,41,144,HARD,17.0,0.803922,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6446,2022-06-12:4:47:9081.336000,2022-06-12,4,47,144,HARD,26.0,0.921569,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6444,2022-06-12:4:45:8864.795000,2022-06-12,4,45,144,HARD,24.0,0.882353,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6443,2022-06-12:4:44:8756.556000,2022-06-12,4,44,144,HARD,23.0,0.862745,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6442,2022-06-12:4:43:8647.282000,2022-06-12,4,43,144,HARD,22.0,0.843137,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6440,2022-06-12:4:41:8431.092000,2022-06-12,4,41,144,HARD,20.0,0.803922,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6439,2022-06-12:4:40:8322.922000,2022-06-12,4,40,144,HARD,19.0,0.784314,NaN,0.0,tree__max_depth_6,0.941621,True,FP


FN window examples (lowest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
2371,2022-04-24:11:17:5454.543000,2022-04-24,11,17,6,INTERMEDIATE,16.0,0.269841,2022-04-24:11:pit:5547.006000,1.0,tree__max_depth_6,0.0,False,FN
2528,2022-04-24:20:17:5482.711000,2022-04-24,20,17,6,INTERMEDIATE,16.0,0.269841,2022-04-24:20:pit:5577.135000,1.0,tree__max_depth_6,0.0,False,FN
2580,2022-04-24:22:15:5300.413000,2022-04-24,22,15,6,INTERMEDIATE,14.0,0.238095,2022-04-24:22:pit:5586.428000,1.0,tree__max_depth_6,0.0,False,FN
2581,2022-04-24:22:16:5396.382000,2022-04-24,22,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:22:pit:5586.428000,1.0,tree__max_depth_6,0.0,False,FN
2582,2022-04-24:22:17:5491.489000,2022-04-24,22,17,6,INTERMEDIATE,16.0,0.269841,2022-04-24:22:pit:5586.428000,1.0,tree__max_depth_6,0.0,False,FN
2633,2022-04-24:23:14:5216.232000,2022-04-24,23,14,6,INTERMEDIATE,13.0,0.222222,2022-04-24:23:pit:5501.637000,1.0,tree__max_depth_6,0.0,False,FN
2635,2022-04-24:23:16:5407.094000,2022-04-24,23,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:23:pit:5501.637000,1.0,tree__max_depth_6,0.0,False,FN
2527,2022-04-24:20:16:5389.244000,2022-04-24,20,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:20:pit:5577.135000,1.0,tree__max_depth_6,0.0,False,FN


Missed events: 313 / 370


,event_id,race_id,driver_id,positive_windows,alerted_positive_windows,caught,missed_event
0,2022-03-20:10:pit:5169.339000,2022-03-20,10,3,0,False,True
2,2022-03-20:11:pit:5253.086000,2022-03-20,11,3,0,False,True
4,2022-03-20:11:pit:8037.416000,2022-03-20,11,3,0,False,True
5,2022-03-20:14:pit:4871.171000,2022-03-20,14,3,0,False,True
8,2022-03-20:16:pit:5239.419000,2022-03-20,16,3,0,False,True
10,2022-03-20:16:pit:8333.486000,2022-03-20,16,3,0,False,True
13,2022-03-20:18:pit:8372.487000,2022-03-20,18,2,0,False,True
14,2022-03-20:1:pit:5143.479000,2022-03-20,1,3,0,False,True


### Find the cases behind your claim

Set CONTEXT_COLUMN to a supplied context column (for example compound or circuit_id), CONTEXT_VALUE to an actual value of the same type shown in your predictions, and ERROR_KIND to FP or FN. The summary uses all rows in that context, so its denominators do not accidentally count only errors. `cases` contains every matching case; the display previews eight. For three patterns you can repeat the query with recorded settings, or copy this query block within this section. Keep each setting and relevant output alongside its written claim.

For a whole-event claim, copy an event_id from the missed-event table into EVENT_TO_INSPECT. The second query retrieves all its positive windows, including their opportunity_id values, without a context filter. An event is missed only if none alerts. Unknown IDs return no rows and are not evidence of a missed event.


In [11]:
from lab03_support_v3 import context_query, positive_windows_for_event

# Pattern 1: false alerts at Baku (circuit_id is a string in the predictions)
CONTEXT_COLUMN = 'circuit_id'
CONTEXT_VALUE = '144'
ERROR_KIND = 'FP'

if predictions is None:
    print('Run your additional candidate first; its frozen predictions are needed.')
elif CONTEXT_COLUMN is None or CONTEXT_VALUE is None:
    print('Set the context column/value above to inspect your own error evidence.')
else:
    context_summary, cases = context_query(predictions, CONTEXT_COLUMN, CONTEXT_VALUE, ERROR_KIND)
    display(context_summary)
    print('Matching cases:', len(cases), '(first eight shown)')
    display(cases.head(8))


,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,circuit_id,144,720,9,51,79,581,60,632,88,0.85,0.080696,0.897727


Matching cases: 51 (first eight shown)


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
6015,2022-06-12:1:46:8888.147000,2022-06-12,1,46,144,HARD,12.0,0.901961,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6016,2022-06-12:1:47:8995.010000,2022-06-12,1,47,144,HARD,13.0,0.921569,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6050,2022-06-12:10:39:8191.055000,2022-06-12,10,39,144,HARD,29.0,0.764706,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6051,2022-06-12:10:40:8299.781000,2022-06-12,10,40,144,HARD,30.0,0.784314,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6052,2022-06-12:10:41:8408.768000,2022-06-12,10,41,144,HARD,31.0,0.803922,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6053,2022-06-12:10:42:8518.030000,2022-06-12,10,42,144,HARD,32.0,0.823529,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6054,2022-06-12:10:43:8627.385000,2022-06-12,10,43,144,HARD,33.0,0.843137,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6055,2022-06-12:10:44:8738.056000,2022-06-12,10,44,144,HARD,34.0,0.862745,NaN,0.0,tree__max_depth_6,0.941621,True,FP


In [12]:
# Pattern 1, extra view: are the false alerts spread out or repeated on the same drivers?
print('Races:', cases.race_id.value_counts().to_dict())
display(cases.groupby('driver_id').agg(false_alerts=('lap_number', 'size'),
                                       first_lap=('lap_number', 'min'), last_lap=('lap_number', 'max'))
        .sort_values('false_alerts', ascending=False))


Races: {'2022-06-12': 51}


,false_alerts,first_lap,last_lap
driver_id,,,
10,7,39,47
4,7,39,47
5,7,39,46
14,6,41,47
47,6,41,46
23,5,42,46
6,5,39,43
63,3,45,47
1,2,46,47


In [13]:
# Pattern 2 (copy of the query block): false alerts on lap 2
P2_COLUMN, P2_VALUE, P2_KIND = 'lap_number', 2, 'FP'
p2_summary, p2_cases = context_query(predictions, P2_COLUMN, P2_VALUE, P2_KIND)
display(p2_summary)
print('Matching cases:', len(p2_cases), '| by race:', p2_cases.race_id.value_counts().to_dict())
display(p2_cases.head(8))
print('Lap-2 windows and alerts per race:')
display(predictions.loc[predictions.lap_number.eq(2)].groupby('race_id')
        .agg(windows=('y', 'size'), alerts=('alert', 'sum'), positive_windows=('y', 'sum')))


,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,lap_number,2,171,0,40,5,126,40,166,5,1.0,0.240964,1.0


Matching cases: 40 | by race: {'2022-03-20': 20, '2022-05-08': 20}


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
0,2022-03-20:1:2:3953.220000,2022-03-20,1,2,63,SOFT,4.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
42,2022-03-20:10:2:3959.543000,2022-03-20,10,2,63,SOFT,4.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
78,2022-03-20:11:2:3957.189000,2022-03-20,11,2,63,SOFT,4.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
119,2022-03-20:14:2:3960.690000,2022-03-20,14,2,63,SOFT,4.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
160,2022-03-20:16:2:3952.027000,2022-03-20,16,2,63,SOFT,1.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
205,2022-03-20:18:2:3966.562000,2022-03-20,18,2,63,SOFT,1.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
247,2022-03-20:20:2:3956.430000,2022-03-20,20,2,63,SOFT,1.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP
290,2022-03-20:22:2:3963.929000,2022-03-20,22,2,63,SOFT,1.0,0.035088,NaN,0.0,tree__max_depth_6,0.843174,True,FP


Lap-2 windows and alerts per race:


,windows,alerts,positive_windows
race_id,,,
2022-03-20,20,20,0.0
2022-03-27,18,0,0.0
2022-04-10,19,0,1.0
2022-05-08,20,20,0.0
2022-05-22,18,0,0.0
2022-05-29,17,0,1.0
2022-06-12,20,0,1.0
2022-06-19,20,0,2.0
2022-07-10,19,0,0.0


In [14]:
# Pattern 3 (copy of the query block): missed positive windows on wet-weather tyres
P3_COLUMN, P3_VALUE, P3_KIND = 'compound', 'INTERMEDIATE', 'FN'
p3_summary, p3_cases = context_query(predictions, P3_COLUMN, P3_VALUE, P3_KIND)
wet_summary, wet_cases = context_query(predictions, 'compound', 'WET', 'FN')
display(pd.concat([p3_summary, wet_summary], ignore_index=True))
print('INTERMEDIATE FN cases:', len(p3_cases), '| by race:', p3_cases.race_id.value_counts().to_dict())
display(p3_cases.head(8))


,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,compound,INTERMEDIATE,377,0,0,86,291,0,291,86,NaN,0.0,1.0
1,compound,WET,229,0,0,47,182,0,182,47,NaN,0.0,1.0


INTERMEDIATE FN cases: 86 | by race: {'2022-04-24': 56, '2022-05-29': 30}


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
2265,2022-04-24:1:16:5355.778000,2022-04-24,1,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:1:pit:5630.145000,1.0,tree__max_depth_6,0.0,False,FN
2266,2022-04-24:1:17:5447.857000,2022-04-24,1,17,6,INTERMEDIATE,16.0,0.269841,2022-04-24:1:pit:5630.145000,1.0,tree__max_depth_6,0.0,False,FN
2267,2022-04-24:1:18:5539.297000,2022-04-24,1,18,6,INTERMEDIATE,17.0,0.285714,2022-04-24:1:pit:5630.145000,1.0,tree__max_depth_6,0.0,False,FN
2315,2022-04-24:10:14:5213.979000,2022-04-24,10,14,6,INTERMEDIATE,13.0,0.222222,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2316,2022-04-24:10:15:5308.891000,2022-04-24,10,15,6,INTERMEDIATE,14.0,0.238095,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2317,2022-04-24:10:16:5404.064000,2022-04-24,10,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2369,2022-04-24:11:15:5269.935000,2022-04-24,11,15,6,INTERMEDIATE,14.0,0.238095,2022-04-24:11:pit:5547.006000,1.0,tree__max_depth_6,0.0,False,FN
2370,2022-04-24:11:16:5362.221000,2022-04-24,11,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:11:pit:5547.006000,1.0,tree__max_depth_6,0.0,False,FN


In [15]:
# Pattern 3: one whole entry from the Imola wet race
EVENT_TO_INSPECT = '2022-04-24:10:pit:5498.635000'

if predictions is None:
    print('Run your additional candidate first.')
elif EVENT_TO_INSPECT is None:
    print('Set an event_id above when checking a completely missed event.')
else:
    event_windows = positive_windows_for_event(predictions, EVENT_TO_INSPECT)
    display(event_windows)
    if event_windows.empty:
        print('No positive windows found: check the ID; do not call this a missed event.')
    else:
        print('Positive windows:', len(event_windows), 'Alerted:', int(event_windows.alert.sum()))
        print('Completely missed event:', not bool(event_windows.alert.any()))


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
2315,2022-04-24:10:14:5213.979000,2022-04-24,10,14,6,INTERMEDIATE,13.0,0.222222,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2316,2022-04-24:10:15:5308.891000,2022-04-24,10,15,6,INTERMEDIATE,14.0,0.238095,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2317,2022-04-24:10:16:5404.064000,2022-04-24,10,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN


Positive windows: 3 Alerted: 0
Completely missed event: True


In [16]:
# Pattern 3, whole-event check: every entry that has at least one positive window on INTERMEDIATE/WET.
# For each of those entries we look at ALL its positive windows (any compound), same as the event query.
wet_event_ids = predictions.loc[predictions.y.eq(1) & predictions.compound.isin(['INTERMEDIATE', 'WET']),
                                'event_id'].unique()
wet_events = (predictions.loc[predictions.y.eq(1) & predictions.event_id.isin(wet_event_ids)]
              .groupby('event_id')
              .agg(race_id=('race_id', 'first'), positive_windows=('y', 'size'),
                   alerted_positive_windows=('alert', 'sum'),
                   compounds=('compound', lambda s: '/'.join(sorted(set(s))))))
wet_events['missed_event'] = wet_events.alerted_positive_windows.eq(0)
print('Entries with wet-tyre positive windows:', len(wet_events),
      '| completely missed:', int(wet_events.missed_event.sum()))
display(wet_events.groupby(['race_id', 'compounds']).agg(entries=('missed_event', 'size'),
                                                        missed=('missed_event', 'sum')))

# Which candidate leaves do those wet positive windows fall in, and what did those leaves see in training?
# (uses dev_leaf / train_leaf from the section 4 check cell; predictions keep the same row order as dev)
wet_pos = (predictions.y.eq(1) & predictions.compound.isin(['INTERMEDIATE', 'WET'])).to_numpy()
wet_leaves = (predictions.loc[wet_pos, ['race_id', 'compound', 'score']].assign(leaf=dev_leaf[wet_pos])
              .groupby(['race_id', 'compound', 'leaf']).agg(positive_windows=('score', 'size'), score=('score', 'first')))
leaf_ids = wet_leaves.index.get_level_values('leaf')
wet_leaves['train_windows'] = [int((train_leaf == leaf).sum()) for leaf in leaf_ids]
wet_leaves['train_positive'] = [int(train.y[train_leaf == leaf].sum()) for leaf in leaf_ids]
wet_leaves['top_train_race'] = [train.race_id[train_leaf == leaf].value_counts().index[0] for leaf in leaf_ids]
wet_leaves['top_train_race_windows'] = [int(train.race_id[train_leaf == leaf].value_counts().iloc[0]) for leaf in leaf_ids]
display(wet_leaves)

print('All missed entries (candidate, dev):', int(event_errors.missed_event.sum()), '/', len(event_errors))
display(event_errors.groupby('race_id').agg(entries=('event_id', 'size'), missed=('missed_event', 'sum')))


Entries with wet-tyre positive windows: 46 | completely missed: 46


entries  missed
race_id    compounds                    
2022-04-24 INTERMEDIATE       19      19
2022-05-29 INTERMEDIATE       11      11
           WET                16      16

positive_windows     score  train_windows  \
race_id    compound     leaf                                              
2022-04-24 INTERMEDIATE 6                    1  0.214145           3930   
                        20                   1  0.422984            899   
                        27                  51  0.000000            276   
                        97                   3  0.472096           3345   
2022-05-29 INTERMEDIATE 64                  15  0.144256           6417   
                        67                  15  0.301889           3921   
           WET          6                    1  0.214145           3930   
                        20                   5  0.422984            899   
                        24                  41  0.736467           3461   

                              train_positive top_train_race  \
race_id    compound     leaf                                  
2022-04-24 INTERMEDIATE 6                100     2019-05-12   
                        20                59     2018-10-07   
                        27                 0     2021-10-10   
                        97               264     2021-05-02   
2022-05-29 INTERMEDIATE 64               102     2021-10-24   
                        67               156     2020-11-29   
           WET          6                100     2019-05-12   
                        20                59     2018-10-07   
                        24               731     2019-10-13   

                              top_train_race_windows  
race_id    compound     leaf                          
2022-04-24 INTERMEDIATE 6                        170  
                        20                        67  
                        27                       228  
                        97                       152  
2022-05-29 INTERMEDIATE 64                       270  
                        67                       186  
           WET          6                        170  
                        20                        67  
                        24                       194

All missed entries (candidate, dev): 313 / 370


,entries,missed
race_id,,
2022-03-20,59,35
2022-03-27,19,15
2022-04-10,21,17
2022-04-24,25,25
2022-05-08,27,15
2022-05-22,53,51
2022-05-29,30,30
2022-06-12,30,27
2022-06-19,32,28


### What counts as a separate pattern

The same error type in different contexts can count as separate patterns when each entry states its context rule, gives its own count/denominator and traceable case, and explains a consequence and next check. Three cases of the same error in the same context count as one pattern. Changing only a context name without evidence and interpretation is incomplete. You do not need three different causes, independent groups or statistical significance. Declare overlap; do not add overlapping counts as distinct errors.

Fictional examples for interpreting the requirement (these are not Rival results):

- Acceptable entry A: “For context `compound == TOY_A`, 2 of 6 alerts are false (FP/alerts), including `toy-A-4`. These alerts require checking a rival that does not enter within the horizon; next check the neighbouring windows to see whether alerts repeat around the same situation. The six-alert sample is small.”
- Acceptable entry B: “For context `compound == TOY_B`, 3 of 10 alerts are false, including `toy-B-2`. The review burden also occurs here; next inspect the recorded timing around those cases before proposing a cause.” With their separate supporting rows, A and B count as two patterns even though both are FP. Your submission must provide the actual rows behind its own claims.
- Incomplete: “TOY_A has false alerts; TOY_B has false alerts; TOY_C has false alerts.” Context names alone supply neither counts/denominators nor cases, consequences or checks. Three IDs from TOY_A under the same rule also describe one pattern.


### Your three entries

All on the frozen `tree__max_depth_6` dev predictions (threshold 0.790). Settings and outputs: pattern 1 `context-query` and `query-pattern1-repeats`; pattern 2 `query-pattern2`; pattern 3 `query-pattern3`, `event-query`, `query-pattern3-events`; leaf origins in the section 4 check cell.

1. **Pattern / rule:** false alerts late in the race at Baku, `circuit_id == '144'`, FP. **Evidence:** 51 of 60 alerts are false (FP/alerts = 0.85; FPR 51/632 = 0.081), all in the 2022-06-12 race, laps 39–47, on 11 drivers; drivers 4, 5 and 10 get 7 false alerts each. All come from leaf 113, learned from 56 train windows, 53 of them from 2021-06-06 (measured); that those were safety-car / red-flag stops is a hypothesis. **opportunity_id:** `2022-06-12:10:39:8191.055000`. **Consequence / next check:** in the closing laps (39–47 of 51) the pit wall gets repeated "about to stop" calls on the same rivals that never happen, which teaches people to ignore the alert. Next: line up that leaf's 2021 windows with the neutralisation laps of that race. Circuit 149 (96/106 alerts false, 51 of those FP from leaf 83) looks like the same mechanism, so we don't count it separately. Also, in train `circuit_id 149` covers both 2020-09-13 and 2021-12-05, which we think were different venues; the mapping should be checked.

2. **Pattern / rule:** false alerts at the race start, `lap_number == 2`, FP. **Evidence:** 40 of 40 lap-2 alerts are false (FP/alerts = 1.0; FPR 40/166 = 0.24): every car on lap 2 in Bahrain (20/20) and Miami (20/20), none in other races. All are in leaf 18 (`prior_pit_entries = 0`, `progress <= 0.035`, `pace_median_s > 91.5`, `pace_n <= 2`), learned from 50 train windows, 48% on wet tyres (measured). Hypothesis: it learned early stops from wet starts and now fires on dry starts with a slow first lap; `progress <= 0.035` only lets lap 2 in for long races (2/57), which would explain why only these two races are hit. **opportunity_id:** `2022-03-20:1:2:3953.220000`. **Consequence / next check:** 20 false alerts at once, at a busy moment of the race. Next: list leaf 18's train windows by race and lap to confirm or reject the wet-start reading.

3. **Pattern / rule:** missed entries on wet tyres, `compound == 'INTERMEDIATE'` (and `'WET'`), FN checked at event level. **Evidence:** 86/86 positive INTERMEDIATE windows are quiet (miss_rate 1.0; 0 alerts in 377 windows, so precision is undefined), and WET is the same, 47/47. Checking *all* positive windows of each entry: 46 entries have wet-tyre positive windows (Imola 19, Monaco 27) and 46/46 are completely missed, out of 313 missed entries overall. Measured: 51 of the 56 Imola positive windows fall in leaf 27 (score 0.0), learned from 276 INTERMEDIATE windows with 0 stops, 228 of them from 2021-10-10; 41 of the 47 Monaco WET positives are in the 794-window block at 0.736, just under the threshold (the same tie as in section 4). Why those training windows had no stops is a hypothesis. **opportunity_id:** `2022-04-24:10:14:5213.979000`, from entry `2022-04-24:10:pit:5498.635000` (3 positive windows, 0 alerted). **Consequence / next check:** the alert is silent exactly when the whole field changes tyres for the conditions. Wet running is only 4.4% of train windows, so the sample is small. Next: count wet train windows and stops per race, and score these 46 entries with forest_sqrt at its fixed threshold.

**Overlap and other checks.** Pattern 3 rows are all at Imola and Monaco (circuits 6 and 22), which also get 0 alerts in their dry windows. The three patterns are in different races, so no window is counted twice, but we don't add their counts as independent. Dry races also have many misses (Silverstone: 0 alerts, 32/32 entries missed; Spain: 4 alerts, 51/53 missed). We did not find a cause, so we report it as a gap, not as a fourth pattern.


In [17]:
manifest = export_run(comparison, policies, change, ROOT, predictions=predictions)
print('Additional experiment executed:', manifest['experiment_complete'])
print('This flag does NOT check your reasoning, three patterns, memo or AI record.')
print('Systems:', comparison.model.nunique(), '| train/dev rows:', len(comparison))
print('Generated comparison_table.md and all five outputs/ files. No manual transcription needed.')


Additional experiment executed: True
This flag does NOT check your reasoning, three patterns, memo or AI record.
Systems: 6 | train/dev rows: 12
Generated comparison_table.md and all five outputs/ files. No manual transcription needed.


## 6. Memo and submission

Complete memo.md and PROMPTS.md. The comparison table is generated automatically.
Check CHECKLIST.md, restart + run all, save visible outputs and push the final commit.
Completed comparison: six systems / twelve train-dev rows, plus your additional candidate's diagnostics.
Canvas text entry: both names, accessible repository URL, final commit and notebook path.
The execution flag does not certify your explanation or completed documents.


## Use your feedback next time

We did not have a Lab 2 comment that fits this lab directly, so we picked a weakness ourselves. Our expectation for the deeper tree was based on a rule of thumb (more leaves = finer threshold steps) that we checked only on the parent, not on how the new leaves would be filled by 2022 data, and it was wrong. For the capstone we want to write down which assumption the expectation depends on and how we would see it fail, before running anything.
